# Data Engineer Certification - Practical Exam - Supplement Experiments

1001-Experiments makes personalized supplements tailored to individual health needs.

1001-Experiments aims to enhance personal health by using data from wearable devices and health apps.

This data, combined with user feedback and habits, is used to analyze and refine the effectiveness of the supplements provided to the user through multiple small experiments.

The data engineering team at 1001-Experiments plays a crucial role in ensuring the collected health and activity data from thousands of users is accurately organized and integrated with the data from supplement usage. 

This integration helps 1001-Experiments provide more targeted health and wellness recommendations and improve supplement formulations.


## Task

1001-Experiments currently has the following four datasets with four months of data:
 - "user_health_data.csv" which logs daily health metrics, habits and data from wearable devices,
 - "supplement_usage.csv" which records details on supplement intake per user,
 - "experiments.csv" which contains metadata on experiments, and
 - "user_profiles.csv" which contains demographic and contact information of the users.

Each dataset contains unique identifiers for users and/or their supplement regimen.

The developers and data scientsits currently manage code that cross-references all of these data sources separately, which is cumbersome and error-prone.

Your manager has asked you to write a Python function that cleans and merges these datasets into a single dataset.

The final dataset should provide a comprehensive view of each user's health metrics, supplement usage, and demographic information.

- To test your code, your manager will run only the code `merge_all_data('user_health_data.csv', 'supplement_usage.csv', 'experiments.csv', 'user_profiles.csv')`
- Your `merge_all_data` function must return a DataFrame, with columns as described below.
- All columns must accurately match the descriptions provided below, including names.


## Data

The provided data is structured as follows:

![database schema](schema.png)

The function you write should return data as described below.

There should be a unique row for each daily entry combining health metrics and supplement usage.

Where missing values are permitted, they should be in the default Python format unless stated otherwise.

| Column Name        | Description |
|--------------------|-------------|
| user_id            | Unique identifier for each user. </br>There should not be any missing values. |
| date               | The date the health data was recorded or the supplement was taken, in date format. </br>There should not be any missing values. |
| email              | Contact email of the user. </br>There should not be any missing values. |
| user_age_group  | The age group of the user, one of: 'Under 18', '18-25', '26-35', '36-45', '46-55', '56-65', 'Over 65' or 'Unknown' where the age is missing.|
| experiment_name    | Name of the experiment associated with the supplement usage. </br>Missing values for users that have user health data only is permitted. |
| supplement_name    | The name of the supplement taken on that day. Multiple entries are permitted. </br>Days without supplement intake should be encoded as 'No intake'. |
| dosage_grams       | The dosage of the supplement taken in grams. Where the dosage is recorded in mg it should be converted by division by 1000.</br>Missing values for days without supplement intake are permitted. |
| is_placebo         | Indicator if the supplement was a placebo (true/false). </br>Missing values for days without supplement intake are permitted. |
| average_heart_rate | Average heart rate as recorded by the wearable device. </br>Missing values are permitted. |
| average_glucose    | Average glucose levels as recorded on the wearable device. </br>Missing values are permitted. |
| sleep_hours        | Total sleep in hours for the night preceding the current day’s log. </br>Missing values are permitted. |
| activity_level     | Activity level score between 0-100. </br>Missing values are permitted. |

## Objetivo

Crear una función `merge_all_data()` que lea, limpie y combine cuatro archivos CSV relacionados con datos de salud, uso de suplementos, experimentos y perfiles de usuarios.

El resultado final debe ser un DataFrame único con métricas de salud, información del suplemento, experimento asociado y datos demográficos del usuario.

# DE601P - Supplement Experiments

## Objetivo del proyecto

El objetivo de este proyecto es construir una función llamada `merge_all_data()` que permita integrar cuatro fuentes de datos relacionadas con salud, uso de suplementos, experimentos y perfiles de usuarios.

La función debe leer los archivos CSV, limpiar y transformar los datos necesarios, combinar las tablas correctamente y devolver un DataFrame final consolidado.

Este DataFrame permite tener una vista única por usuario y fecha, incluyendo métricas de salud, información del suplemento consumido, experimento asociado y datos demográficos del usuario.

In [10]:
# Importamos librerías necesarias
import pandas as pd
import numpy as np


# Definimos la función principal del proyecto
def merge_all_data(user_health_file, supplement_usage_file, experiments_file, user_profiles_file):

    # 1. Cargar archivos CSV
    user_health = pd.read_csv(user_health_file)
    supplement_usage = pd.read_csv(supplement_usage_file)
    experiments = pd.read_csv(experiments_file)
    user_profiles = pd.read_csv(user_profiles_file)

    # 2. Convertir columnas date a formato fecha
    user_health["date"] = pd.to_datetime(user_health["date"]).dt.date
    supplement_usage["date"] = pd.to_datetime(supplement_usage["date"]).dt.date

    # 3. Renombrar name de experiments como experiment_name
    experiments = experiments.rename(columns={"name": "experiment_name"})

    # 4. Unir supplement_usage con experiments para agregar experiment_name
    supplement_usage = supplement_usage.merge(
        experiments[["experiment_id", "experiment_name"]],
        on="experiment_id",
        how="left"
    )

    # 5. Convertir dosage a gramos
    supplement_usage["dosage_unit"] = supplement_usage["dosage_unit"].str.lower()

    supplement_usage["dosage_grams"] = np.where(
        supplement_usage["dosage_unit"].isin(["mg", "milligram", "milligrams"]),
        supplement_usage["dosage"] / 1000,
        supplement_usage["dosage"]
    )

    # 6. Unir datos de salud con datos de suplementos
    merged = user_health.merge(
        supplement_usage[
            [
                "user_id",
                "date",
                "experiment_name",
                "supplement_name",
                "dosage_grams",
                "is_placebo"
            ]
        ],
        on=["user_id", "date"],
        how="left"
    )

    # 7. Unir con perfiles de usuario
    merged = merged.merge(
        user_profiles[["user_id", "email", "age"]],
        on="user_id",
        how="left"
    )

    # 8. Crear grupos de edad
    bins = [-np.inf, 17, 25, 35, 45, 55, 65, np.inf]
    labels = ["Under 18", "18-25", "26-35", "36-45", "46-55", "56-65", "Over 65"]

    merged["user_age_group"] = pd.cut(
        merged["age"],
        bins=bins,
        labels=labels
    )

    merged["user_age_group"] = merged["user_age_group"].astype("object")
    merged["user_age_group"] = merged["user_age_group"].fillna("Unknown")

    # 9. Marcar días sin consumo de suplemento
    merged["supplement_name"] = merged["supplement_name"].fillna("No intake")

    # 10. Limpiar sleep_hours extrayendo solo el número y convirtiendo a float
    merged["sleep_hours"] = (
        merged["sleep_hours"]
        .astype(str)
        .str.extract(r"(\d+\.?\d*)")[0]
        .astype(float)
    )

    # 11. Convertir is_placebo a booleano permitiendo valores faltantes
    merged["is_placebo"] = merged["is_placebo"].astype("boolean")

    # 12. Seleccionar columnas finales en el orden solicitado
    final_df = merged[
        [
            "user_id",
            "date",
            "email",
            "user_age_group",
            "experiment_name",
            "supplement_name",
            "dosage_grams",
            "is_placebo",
            "average_heart_rate",
            "average_glucose",
            "sleep_hours",
            "activity_level"
        ]
    ]

    # 13. Retornar resultado final
    return final_df

## Validación del resultado

Después de ejecutar la función, se valida que:

- El resultado sea un DataFrame.
- Las columnas estén en el orden correcto.
- Las columnas obligatorias no tengan valores faltantes.
- `sleep_hours` quede como número decimal.
- `is_placebo` quede como booleano.
- Los días sin suplemento aparezcan como `No intake`.

In [11]:
# Ejecutar función con los archivos del proyecto
result = merge_all_data(
    "user_health_data.csv",
    "supplement_usage.csv",
    "experiments.csv",
    "user_profiles.csv"
)

# Revisar primeras filas
print(result.head())

# Revisar estructura, tipos de datos y valores no nulos
print(result.info())

# Revisar cantidad de valores faltantes por columna
print(result.isna().sum())

# Revisar nombres y orden de columnas
print(result.columns.tolist())

# Revisar tamaño final del DataFrame
print(result.shape)

                                user_id  ... activity_level
0  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              1
1  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              3
2  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              1
3  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              1
4  c6ae338a-9f95-481c-a88d-24a58bc8fc71  ...              1

[5 rows x 12 columns]
<class 'pandas.core.frame.DataFrame'>
Int64Index: 2721 entries, 0 to 2720
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             2721 non-null   object 
 1   date                2721 non-null   object 
 2   email               2721 non-null   object 
 3   user_age_group      2721 non-null   object 
 4   experiment_name     2000 non-null   object 
 5   supplement_name     2721 non-null   object 
 6   dosage_grams        2000 non-null   float64
 7   is_placebo          2000 non-null   boolean
 8   average_heart_r

## Pasos desarrollados

### 1. Cargar archivos CSV

Se cargan los cuatro archivos del proyecto usando `pd.read_csv()`:

- `user_health_data.csv`
- `supplement_usage.csv`
- `experiments.csv`
- `user_profiles.csv`

Cada archivo se transforma en un DataFrame para poder trabajarlo con pandas.

---

### 2. Convertir columnas de fecha

Las columnas `date` de `user_health_data` y `supplement_usage` se convierten a formato fecha.

Esto es necesario porque ambas tablas se unen usando `user_id` y `date`.

---

### 3. Renombrar columna de experimentos

En la tabla `experiments`, la columna `name` se renombra como `experiment_name`.

Esto permite que el resultado final tenga un nombre de columna más claro y alineado con los requisitos del proyecto.

---

### 4. Unir suplementos con experimentos

La tabla `supplement_usage` se une con `experiments` usando `experiment_id`.

Con esta unión se agrega el nombre del experimento asociado a cada suplemento.

---

### 5. Convertir dosis a gramos

La columna `dosage` puede venir en distintas unidades.

Cuando la unidad está en miligramos (`mg`), se divide por 1000 para convertirla a gramos.

El resultado se guarda en una nueva columna llamada `dosage_grams`.

---

### 6. Unir datos de salud con suplementos

Se toma `user_health_data` como tabla principal, ya que contiene el registro diario de salud de cada usuario.

Luego se agregan los datos de suplementos usando `user_id` y `date`.

Se utiliza un `left merge` para conservar todos los registros de salud, incluso cuando no existe consumo de suplemento ese día.

---

### 7. Unir con perfiles de usuario

Se agrega la información de `user_profiles`, principalmente:

- `email`
- `age`

La unión se realiza usando `user_id`.

---

### 8. Crear grupos de edad

A partir de la columna `age`, se crea una nueva columna llamada `user_age_group`.

Los grupos definidos son:

- `Under 18`
- `18-25`
- `26-35`
- `36-45`
- `46-55`
- `56-65`
- `Over 65`
- `Unknown`

Cuando la edad está vacía, se asigna el valor `Unknown`.

---

### 9. Marcar días sin consumo de suplemento

Cuando un usuario tiene registro de salud, pero no tomó suplementos ese día, la columna `supplement_name` queda vacía después del merge.

Estos casos se reemplazan por `No intake`.

---

### 10. Limpiar la columna sleep_hours

La columna `sleep_hours` puede venir como texto, por ejemplo `8.0h` o `8.0H`.

Se extrae solo el número y se convierte a tipo `float`.

Esto permite usar la columna para análisis numéricos.

---

### 11. Convertir is_placebo a booleano

La columna `is_placebo` se convierte a tipo booleano.

Esto permite representar correctamente si un suplemento fue placebo (`True`) o no (`False`).

Los valores faltantes se mantienen porque corresponden a días sin consumo de suplemento.

---

### 12. Seleccionar columnas finales

Se seleccionan únicamente las columnas requeridas por el proyecto y en el orden solicitado:

- `user_id`
- `date`
- `email`
- `user_age_group`
- `experiment_name`
- `supplement_name`
- `dosage_grams`
- `is_placebo`
- `average_heart_rate`
- `average_glucose`
- `sleep_hours`
- `activity_level`

---

### 13. Retornar resultado final

La función retorna un DataFrame final consolidado con 12 columnas.

Este resultado queda listo para análisis, validación o uso posterior en un pipeline de datos.

## Conclusión

En este proyecto se desarrolló un proceso ETL usando Python y pandas para integrar datos provenientes de cuatro archivos distintos.

El trabajo incluyó carga de datos, limpieza de fechas, conversión de unidades, tratamiento de valores faltantes, creación de variables categóricas y combinación de tablas mediante claves comunes.

El resultado final es un DataFrame limpio, estructurado y validado, que permite analizar de forma centralizada la relación entre métricas de salud, consumo de suplementos, experimentos y perfiles de usuarios.

Este proyecto demuestra habilidades prácticas de ingeniería de datos como integración de fuentes, transformación de datos, limpieza, manejo de tipos y construcción de una función reutilizable.